In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go


from CausalSurv.data.CV_online_data_utils import ESMEOnlineDataModuleCV

PROJECT_ROOT = Path.cwd().parent
RUN_SEED = 42  # Define your run seed here
MODEL_PATH = ...  # Define your model path here

## Load data

In [ ]:
data_module = ESMEOnlineDataModuleCV(
    data_dir=str(PROJECT_ROOT / "data"),
    subtype="HR+HER2-",
    n_lines=4,
    n_intervals=5,
    batch_size=256,
    split_seed=RUN_SEED,
)
data_module.prepare_data()
data_module.setup(stage="test")
test_loader = data_module.test_dataloader()

In [ ]:
XPd, X_static, interval_idx, treatment_idx, time, event, mask, patient_id = next(
    iter(test_loader)
)

In [ ]:
df = data_module.data.copy()
df = df.loc[
    df["usubjid"].isin(patient_id.numpy()),
    ["usubjid", "lineid", "Y_onset_to_death", "Y_death", "Y_death_inline"],
].reset_index(drop=True)
df["usubjid"].nunique()

## Test data distribution

##### Line distribution

In [ ]:
TOTAL_PATIENTS_PER_LINE = mask.sum(dim=0).numpy()
print("Total patients per line:", TOTAL_PATIENTS_PER_LINE)

In [ ]:
cum_mask = np.cumsum(mask.numpy(), axis=1)
transitions_per_line = {}
for i in range(cum_mask.shape[1] - 1):
    for j in range(cum_mask.shape[0]):
        key = (f"line{i + 1}", f"line{i + 2}")
        transitions_per_line[key] = transitions_per_line.get(key, 0) + (
            cum_mask[j, i + 1] > cum_mask[j, i]
        )

events_per_line = df.groupby("lineid")["Y_death_inline"].sum().values
for event_count in range(len(events_per_line)):
    # print(f"Number of events in line {event_count + 1}: {events_per_line[event_count]}")
    transitions_per_line_key = (f"line{event_count + 1}", f"death{event_count + 1}")
    transitions_per_line[transitions_per_line_key] = events_per_line[event_count]

transition_count = [
    v for k, v in transitions_per_line.items() if "line" in k[0] and "line" in k[1]
] + [0]
censored_per_line = TOTAL_PATIENTS_PER_LINE - (
    events_per_line + np.array(transition_count)
)
for censored_count in range(len(censored_per_line)):
    # print(f"Number of censored patients in line {censored_count + 1}: {censored_per_line[censored_count]}")
    transitions_per_line_key = (
        f"line{censored_count + 1}",
        f"censored{censored_count + 1}",
    )
    transitions_per_line[transitions_per_line_key] = censored_per_line[censored_count]

label = []
source = []
target = []
value = []
for transition, count in transitions_per_line.items():
    # print(f"Number of patients transitioning from {transition[0]} to {transition[1]}: {count}")
    src, tgt = transition
    if src not in label:
        label.append(src)
    if tgt not in label:
        label.append(tgt)
    source.append(label.index(src))
    target.append(label.index(tgt))
    value.append(count)

In [ ]:
# --- 2. YOUR EXISTING STYLING LOGIC ---
label_mapping = {
    "line1": "",
    "line2": "",
    "line3": "",
    "line4": "",
    "death1": "Death",
    "death2": "Death",
    "death3": "Death",
    "death4": "Death",
    "censored1": "Censored",
    "censored2": "Censored",
    "censored3": "Censored",
    "censored4": "Censored",
}

readable_labels = [label_mapping.get(lbl, lbl) for lbl in label]

node_colors = []
for lbl in label:
    if "line1" in lbl:
        node_colors.append("#26C6DA")
    elif "line2" in lbl:
        node_colors.append("#0097A7")
    elif "line3" in lbl:
        node_colors.append("#00677A")
    elif "line4" in lbl:
        node_colors.append("#004D5C")
    elif "death" in lbl:
        node_colors.append("#D32F2F")
    elif "censored" in lbl:
        node_colors.append("#9E9E9E")
    else:
        node_colors.append("#BDBDBD")

link_colors = []
for s, t in zip(source, target):
    target_label = label[t]
    if "death" in target_label:
        link_colors.append("rgba(211, 47, 47, 0.3)")
    elif "censored" in target_label:
        link_colors.append("rgba(158, 158, 158, 0.3)")
    else:
        source_color = node_colors[s]
        if source_color == "#26C6DA":
            link_colors.append("rgba(38, 198, 218, 0.4)")
        elif source_color == "#0097A7":
            link_colors.append("rgba(0, 151, 167, 0.4)")
        elif source_color == "#00677A":
            link_colors.append("rgba(0, 103, 122, 0.4)")
        elif source_color == "#004D5C":
            link_colors.append("rgba(0, 77, 92, 0.4)")
        else:
            link_colors.append("rgba(38, 198, 218, 0.4)")

final_node_labels = list(readable_labels)
final_node_colors = list(node_colors)

# 1. Initialize border colors: Start with white for all original nodes
final_node_line_colors = ["white"] * len(node_colors)
final_node_linewidths = [2] * len(node_colors)

final_source = []
final_target = []
final_value = []
final_link_colors = []

for s, t, v, c in zip(source, target, value, link_colors):
    dummy_node_index = len(final_node_labels)

    final_node_labels.append(str(v))
    final_node_colors.append(c)  # Node matches link transparency

    # FIX: Set the border color to match the link color (c)
    # and set width to a small value or 0 to ensure no "white slice"
    final_node_line_colors.append(c)
    final_node_linewidths.append(0)

    # Link Part 1: Source -> Dummy
    final_source.append(s)
    final_target.append(dummy_node_index)
    final_value.append(v)
    final_link_colors.append(c)

    # Link Part 2: Dummy -> Target
    final_source.append(dummy_node_index)
    final_target.append(t)
    final_value.append(v)
    final_link_colors.append(c)

### ------------------------------------------- ###

fig = go.Figure(
    data=[
        go.Sankey(
            node=dict(
                label=final_node_labels,
                pad=0,
                thickness=20,
                align="left",
                color=final_node_colors,
                # Apply the dynamic colors and widths
                line=dict(color=final_node_line_colors, width=0),
            ),
            link=dict(
                source=final_source,
                target=final_target,
                value=final_value,
                color=final_link_colors,
            ),
        )
    ]
)


fig.update_layout(
    title={
        "text": "Patient Transitions Across Treatment Lines",
        "font": {"size": 18, "color": "#2C3E50", "family": "Arial, sans-serif"},
    },
    font=dict(size=16, color="#2C3E50", family="Arial, sans-serif"),
    width=800,
    margin=dict(t=80, b=40, l=40, r=220),  # Extra right margin for labels
    plot_bgcolor="white",
    paper_bgcolor="white",
)

fig.show()

In [ ]:
# Define meaningful labels
label_mapping = {
    "line1": "Line 1",
    "line2": "Line 2",
    "line3": "Line 3",
    "line4": "Line 4",
    "death1": "Death",
    "death2": "Death",
    "death3": "Death",
    "death4": "Death",
    "censored1": "Censored",
    "censored2": "Censored",
    "censored3": "Censored",
    "censored4": "Censored",
}

# Apply new labels
readable_labels = [label_mapping.get(lbl, lbl) for lbl in label]

# Medical/clinical color scheme with higher contrast between lines
node_colors = []
for lbl in label:
    if "line1" in lbl:
        node_colors.append("#26C6DA")  # Light cyan
    elif "line2" in lbl:
        node_colors.append("#0097A7")  # Medium cyan
    elif "line3" in lbl:
        node_colors.append("#00677A")  # Dark cyan
    elif "line4" in lbl:
        node_colors.append("#004D5C")  # Very dark cyan
    elif "death" in lbl:
        node_colors.append("#D32F2F")  # Clinical red
    elif "censored" in lbl:
        node_colors.append("#9E9E9E")  # Neutral gray
    else:
        node_colors.append("#BDBDBD")  # Default gray

# Color links based on their destination (target)
# This makes it easy to see which flows lead to death vs censoring vs continuation
link_colors = []
for s, t in zip(source, target):
    target_label = label[t]
    if "death" in target_label:
        link_colors.append("rgba(211, 47, 47, 0.3)")  # Red for flows to death
    elif "censored" in target_label:
        link_colors.append("rgba(158, 158, 158, 0.3)")  # Gray for flows to censored
    else:
        # Flows between treatment lines - use source color with transparency
        source_color = node_colors[s]
        if source_color == "#26C6DA":
            link_colors.append("rgba(38, 198, 218, 0.4)")
        elif source_color == "#0097A7":
            link_colors.append("rgba(0, 151, 167, 0.4)")
        elif source_color == "#00677A":
            link_colors.append("rgba(0, 103, 122, 0.4)")
        elif source_color == "#004D5C":
            link_colors.append("rgba(0, 77, 92, 0.4)")
        else:
            link_colors.append("rgba(38, 198, 218, 0.4)")  # Default cyan

# Create the improved Sankey diagram
fig = go.Figure(
    data=[
        go.Sankey(
            node=dict(
                label=readable_labels,
                pad=15,
                thickness=20,
                align="left",
                color=node_colors,
                line=dict(color="white", width=2),  # White borders for clarity
            ),
            link=dict(source=source, target=target, value=value, color=link_colors),
        )
    ]
)

fig.update_layout(
    title={
        "text": "Patient Transitions Across Treatment Lines",
        "font": {"size": 18, "color": "#2C3E50", "family": "Arial, sans-serif"},
    },
    font=dict(size=12, color="#2C3E50", family="Arial, sans-serif"),
    width=800,
    margin=dict(t=80, b=40, l=40, r=220),  # Extra right margin for labels
    plot_bgcolor="white",
    paper_bgcolor="white",
)

fig.show()

##### Treatment distribution

In [ ]:
df = data_module.data.copy()
df = df.loc[
    df["usubjid"].isin(patient_id.numpy()),
    ["usubjid", "lineid", "T_treatment_category"],
].reset_index(drop=True)
df["usubjid"].nunique()

In [ ]:
# Optional: Combine tiny categories into 'Other'
top_categories = df["T_treatment_category"].value_counts().nlargest(7).index
df["T_treatment_category"] = df["T_treatment_category"].apply(
    lambda x: x if x in top_categories else "OTHER"
)

In [ ]:
import seaborn as sns

# 1. Data Preparation (Same logic as before)
df_counts = (
    df.groupby(["lineid", "T_treatment_category"]).size().reset_index(name="count")
)
df_counts["percentage"] = df_counts.groupby("lineid")["count"].transform(
    lambda x: (x / x.sum()) * 100
)

# Reshape for a stacked bar chart
pivot_df = df_counts.pivot(
    index="lineid", columns="T_treatment_category", values="percentage"
).fillna(0)

medical_colors = {
    "CT+ANTI-ANGIO": "#d0fda9",  # Lightest (Yellow-Green tint)
    "ET alone": "#bae4bc",
    "ET+TT": "#7bccc4",
    "ET+ANTI-CDK wo CT": "#43a2ca",
    "POLYCT alone": "#0868ac",
    "MONOCT std alone": "#084081",  # Deep Navy
    "OTHER": "#D3D3D3",  # Light Neutral
}

# Update your pivot columns with this map
colors = [medical_colors.get(col, "#D3D3D3") for col in pivot_df.columns]
# Map colors to the columns found in your data
colors = [medical_colors.get(col, "#D3D3D3") for col in pivot_df.columns]

# 3. Create the Plot
plt.style.use("seaborn-v0_8-white")  # Clean white background
fig, ax = plt.subplots(figsize=(10, 6))

pivot_df.plot(
    kind="bar",
    stacked=True,
    color=colors,
    ax=ax,
    width=0.75,
    edgecolor="white",  # White border between segments for "clean" look
    linewidth=0.5,
)

# 4. Professional Styling
ax.set_title(
    "Treatment Mix Evolution across Lines",
    fontsize=14,
    fontweight="bold",
    pad=20,
    loc="left",
)
ax.set_ylabel("Share of Patients (%)", fontsize=11, fontweight="semibold")
ax.set_xlabel("Treatment Line", fontsize=11, fontweight="semibold")
ax.set_ylim(0, 100)

# Clean up axes (remove top and right spines)
sns.despine()

# Format Ticks
plt.xticks(rotation=0)  # Keep line IDs horizontal
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{int(x)}%"))

# Legend placement (Outside the chart)
plt.legend(
    title="Treatment Category",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False,
    fontsize=9,
)

plt.tight_layout()
plt.plot()
plt.savefig("treatment_mix_evolution.png", dpi=300)
plt.show()